# Survival Analysis with Progenetix CNV Data

In this notebook, we build a simple survival analysis example using Progenetix data.

The basic workflow:

1. retrieve sample-level metadata,
2. retreive individual-level data with survival information,
3. retrieve sample-level CNV records,
4. convert CNV records into simple gene-level features,
5. and apply two standard survival analysis methods.

We will use:

- **Kaplan–Meier curves** to compare survival between two groups,
- and a **Cox proportional hazards model** to estimate how a CNV-derived feature is associated with hazard.



## Environment Setup

```bash
conda install -c conda-forge scikit-survival
pip install scikit-survival

In [1]:
import requests
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

from tqdm.auto import tqdm

from sksurv.util import Surv
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

## Download sample metadata from Progenetix

We begin with the Progenetix `sampletable` service.

This service returns sample annotations in tabular form and supports query filters.
For example, the Progenetix documentation shows that TCGA samples can be retrieved with:

- `filters=pgx:cohort-TCGAcancers`

and that:

- `limit=0` means no result limit.


In [2]:
# -----------------------------
# Cohort definition
# -----------------------------
TARGET_TCGA_PROJECT = "pgx:TCGA-LUAD"   

# -----------------------------
# Progenetix sampletable endpoint
# -----------------------------
SAMPLETABLE_URL = "https://progenetix.org/services/sampletable/"

In [3]:
def fetch_sampletable(filters, limit=0, timeout=120):
    # Download a Progenetix sample table as a pandas DataFrame.
    params = {
        "filters": filters,
        "limit": limit,
    }

    response = requests.get(SAMPLETABLE_URL, params=params, timeout=timeout)
    response.raise_for_status()

    print("Status:", response.status_code)
    print("Final URL:", response.url)
    print("Content-Type:", response.headers.get("Content-Type"))

    text = response.text
    df = pd.read_csv(io.StringIO(text), sep="\t")
    return df

In [4]:
sample_table_df = fetch_sampletable(filters=TARGET_TCGA_PROJECT, limit=0)

print("Shape:", sample_table_df.shape)
print("Columns:")
print(sample_table_df.columns.tolist())
sample_table_df.head()

Status: 200
Final URL: https://progenetix.org/services/sampletable/?filters=pgx%3ATCGA-LUAD&limit=0
Content-Type: text/tsv
Shape: (1110, 42)
Columns:
['biosample_id', 'individual_id', 'biosample_name', 'notes', 'histological_diagnosis_id', 'histological_diagnosis_label', 'pathological_stage_id', 'pathological_stage_label', 'biosample_status_id', 'biosample_status_label', 'sample_origin_type_id', 'sample_origin_type_label', 'sampled_tissue_id', 'sampled_tissue_label', 'tnm', 'tumor_grade_id', 'tumor_grade_label', 'age_iso', 'age_days', 'icdo_morphology_id', 'icdo_morphology_label', 'icdo_topography_id', 'icdo_topography_label', 'pubmed_id', 'pubmed_label', 'cellosaurus_id', 'cellosaurus_label', 'cbioportal_id', 'cbioportal_label', 'tcgaproject_id', 'tcgaproject_label', 'cohorts', 'geoprov_id', 'geoprov_city', 'geoprov_country', 'geoprov_iso_alpha3', 'geoprov_long_lat', 'followup_days', 'sex_id', 'sex_label', 'group_id', 'group_label']


,biosample_id,individual_id,biosample_name,notes,histological_diagnosis_id,histological_diagnosis_label,pathological_stage_id,pathological_stage_label,biosample_status_id,biosample_status_label,...,geoprov_id,geoprov_city,geoprov_country,geoprov_iso_alpha3,geoprov_long_lat,followup_days,sex_id,sex_label,group_id,group_label
0,pgxbs-kftvhmjh,pgxind-kftx3l3h,37a8d3a5-baa0-427c-ab43-48ae4b93633b,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27975,Stage IA,EFO:0009656,neoplastic sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN
1,pgxbs-kftvhqda,pgxind-kftx3hag,5a109a3b-8edf-43f6-afb3-20e9ea68d617,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27975,Stage IA,EFO:0009654,reference sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN
2,pgxbs-kftvi8g3,pgxind-kftx3is0,4bee0576-d35f-477b-b974-79cb89c88950,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27977,Stage IIIA,EFO:0009656,neoplastic sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN
3,pgxbs-kftvhmo2,pgxind-kftx3l8h,ef98c575-44c9-4524-8754-45250f60a5c8,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27971,Stage IV,EFO:0009656,neoplastic sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN
4,pgxbs-kftvhrxj,pgxind-kftx3pvf,81bfa60b-0e23-4d92-b82c-7a9c90e76fd7,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27975,Stage IA,EFO:0009654,reference sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN


In [5]:
# Columns we want to keep from the sample table
BIOSAMPLE_COLUMNS = [
    "biosample_id",
    "individual_id",
    "biosample_name",
    "notes",
    "histological_diagnosis_id",
    "histological_diagnosis_label",
    "pathological_stage_id",
    "pathological_stage_label",
    "sample_origin_type_id",
    "sample_origin_type_label",
    "sampled_tissue_id",
    "sampled_tissue_label",
    "tcgaproject_id",
    "tcgaproject_label",
    "icdo_morphology_id",
    "icdo_topography_id",
    "icdo_topography_label",
]

BIOSAMPLE_COLUMNS = [c for c in BIOSAMPLE_COLUMNS if c in sample_table_df.columns]

biosample_df = sample_table_df[BIOSAMPLE_COLUMNS].copy()

print("Biosample-level metadata shape:", biosample_df.shape)
biosample_df.head()

Biosample-level metadata shape: (1110, 17)


,biosample_id,individual_id,biosample_name,notes,histological_diagnosis_id,histological_diagnosis_label,pathological_stage_id,pathological_stage_label,sample_origin_type_id,sample_origin_type_label,sampled_tissue_id,sampled_tissue_label,tcgaproject_id,tcgaproject_label,icdo_morphology_id,icdo_topography_id,icdo_topography_label
0,pgxbs-kftvhmjh,pgxind-kftx3l3h,37a8d3a5-baa0-427c-ab43-48ae4b93633b,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27975,Stage IA,OBI:0001479,specimen from organism,UBERON:0008948,upper lobe of lung,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-81403,pgx:icdot-C34.1,"Upper lobe, lung"
1,pgxbs-kftvhqda,pgxind-kftx3hag,5a109a3b-8edf-43f6-afb3-20e9ea68d617,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27975,Stage IA,OBI:0001479,specimen from organism,UBERON:0000178,blood,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-00000,pgx:icdot-C42.0,Blood
2,pgxbs-kftvi8g3,pgxind-kftx3is0,4bee0576-d35f-477b-b974-79cb89c88950,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27977,Stage IIIA,OBI:0001479,specimen from organism,UBERON:0008948,upper lobe of lung,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-81403,pgx:icdot-C34.1,"Upper lobe, lung"
3,pgxbs-kftvhmo2,pgxind-kftx3l8h,ef98c575-44c9-4524-8754-45250f60a5c8,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27971,Stage IV,OBI:0001479,specimen from organism,UBERON:0008948,upper lobe of lung,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-81403,pgx:icdot-C34.1,"Upper lobe, lung"
4,pgxbs-kftvhrxj,pgxind-kftx3pvf,81bfa60b-0e23-4d92-b82c-7a9c90e76fd7,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27975,Stage IA,OBI:0001479,specimen from organism,UBERON:0000178,blood,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-00000,pgx:icdot-C42.0,Blood


## Build survival metadata by combining biosample- and individual-level information

The Progenetix sample table is useful for cohort selection and biosample-level metadata, but survival-related information is not always available there.

In our case, each biosample is linked to an `individual_id`.  
We therefore use a two-step strategy:

1. retrieve **biosample-level metadata** from the Progenetix sample table,
2. retrieve **individual-level follow-up metadata** from the Progenetix individual endpoint,
3. and merge the two through `individual_id`.

This is important because survival analysis requires at least:

- a follow-up time,
- and an event indicator.

These fields are available in the individual-level record.

In the Progenetix data model, one individual may have more than one biosample.  
Therefore, after merging biosample- and individual-level metadata, we also need to decide how to handle the one-to-many relationship before building a survival model.

In [6]:
individual_ids = (
    biosample_df["individual_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("Number of unique individuals:", len(individual_ids))
print(individual_ids[:10])

Number of unique individuals: 518
['pgxind-kftx3l3h', 'pgxind-kftx3hag', 'pgxind-kftx3is0', 'pgxind-kftx3l8h', 'pgxind-kftx3pvf', 'pgxind-kftx3rlt', 'pgxind-kftx3s6d', 'pgxind-kftx3iot', 'pgxind-kftx3poj', 'pgxind-kftx3oji']


In [7]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})

def fetch_individual_record(individual_id, session=SESSION, timeout=120):
    # Fetch one Progenetix individual record by individual ID.
    url = f"https://progenetix.org/beacon/individuals/{individual_id}/"
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()

In [8]:
def parse_individual_record(data):
    # Parse one Progenetix individual JSON response into a flat dictionary.
    response = data.get("response", {})
    result_sets = response.get("resultSets", [])
    if not result_sets:
        return None

    results = result_sets[0].get("results", [])
    if not results:
        return None

    rec = results[0]

    diseases = rec.get("diseases", [])
    disease_0 = diseases[0] if len(diseases) > 0 else {}

    out = {
        # identifiers
        "individual_id": rec.get("id"),
        "n_diseases": len(diseases),

        # sex
        "sex_id_individual": rec.get("sex", {}).get("id"),
        "sex_label_individual": rec.get("sex", {}).get("label"),

        # top-level survival-related status
        "vital_status": rec.get("vitalStatus", {}).get("status"),

        # top-level info block
        "info_age_at_diagnosis_days": rec.get("info", {}).get("ageAtDiagnosis"),
        "info_days_to_death": rec.get("info", {}).get("daysToDeath"),
        "info_death": rec.get("info", {}).get("death"),
        "info_ethnicity": rec.get("info", {}).get("ethnicity"),
        "info_race": rec.get("info", {}).get("race"),
        "info_year_of_birth": rec.get("info", {}).get("yearOfBirth"),

        # references
        "tcga_case_id": rec.get("references", {}).get("tcgacase", {}).get("id"),
        "tcga_submitter_id": rec.get("references", {}).get("tcgasubmitter", {}).get("id"),

        # disease block 
        "disease_id": disease_0.get("diseaseCode", {}).get("id"),
        "disease_label": disease_0.get("diseaseCode", {}).get("label"),
        "followup_days": disease_0.get("followupDays"),
        "followup_time_iso": disease_0.get("followupTime"),
        "followup_state_id": disease_0.get("followupState", {}).get("id"),
        "followup_state_label": disease_0.get("followupState", {}).get("label"),
        "onset_age_iso": disease_0.get("onset", {}).get("age"),
        "onset_age_days": disease_0.get("onset", {}).get("ageDays"),
        "stage_id_individual": disease_0.get("stage", {}).get("id"),
        "stage_label_individual": disease_0.get("stage", {}).get("label"),
    }

    return out

In [9]:
individual_rows = []
failed_individuals = []

for individual_id in tqdm(individual_ids, desc="Fetching individual records"):
    try:
        data = fetch_individual_record(individual_id)
        row = parse_individual_record(data)
        if row is not None:
            individual_rows.append(row)
        else:
            failed_individuals.append((individual_id, "No result"))
    except Exception as e:
        failed_individuals.append((individual_id, str(e)))

individual_df = pd.DataFrame(individual_rows)

print("Fetched individual rows:", len(individual_df))
print("Failed individual queries:", len(failed_individuals))
individual_df.head()

Fetching individual records:   0%|          | 0/518 [00:00<?, ?it/s]

Fetched individual rows: 518
Failed individual queries: 0


,individual_id,n_diseases,sex_id_individual,sex_label_individual,vital_status,info_age_at_diagnosis_days,info_days_to_death,info_death,info_ethnicity,info_race,...,disease_id,disease_label,followup_days,followup_time_iso,followup_state_id,followup_state_label,onset_age_iso,onset_age_days,stage_id_individual,stage_label_individual
0,pgxind-kftx3l3h,1,NCIT:C16576,female,ALIVE,22471.0,NaN,alive,not hispanic or latino,black or african american,...,NCIT:C3512,Lung Adenocarcinoma,851.0,P28M,EFO:0030041,alive (follow-up status),P61Y6M7D,22469.0,NCIT:C27975,Stage IA
1,pgxind-kftx3hag,1,NCIT:C16576,female,ALIVE,23055.0,NaN,alive,not hispanic or latino,black or african american,...,NCIT:C65197,Lung Adenocarcinoma with Mixed Bronchioloalveo...,1125.0,P37M,EFO:0030041,alive (follow-up status),P63Y1M13D,23053.0,NCIT:C27975,Stage IA
2,pgxind-kftx3is0,1,NCIT:C16576,female,ALIVE,27676.0,NaN,alive,not hispanic or latino,black or african american,...,NCIT:C3512,Lung Adenocarcinoma,30.0,P1M,EFO:0030041,alive (follow-up status),P75Y9M7D,27673.0,NCIT:C27977,Stage IIIA
3,pgxind-kftx3l8h,1,NCIT:C16576,female,DECEASED,14681.0,58.0,dead,not hispanic or latino,black or african american,...,NCIT:C3512,Lung Adenocarcinoma,NaN,NaN,EFO:0030049,dead (follow-up status),P40Y2M10D,14680.0,NCIT:C27971,Stage IV
4,pgxind-kftx3pvf,1,NCIT:C16576,female,DECEASED,22414.0,550.0,dead,not hispanic or latino,white,...,NCIT:C3512,Lung Adenocarcinoma,273.0,P9M,EFO:0030049,dead (follow-up status),P61Y4M11D,22412.0,NCIT:C27975,Stage IA


### Merge Biosample and Individual with individual_id

In [10]:
merged_meta_df = biosample_df.merge(
    individual_df,
    on="individual_id",
    how="left"
)

print("Merged metadata shape:", merged_meta_df.shape)
merged_meta_df.head()

Merged metadata shape: (1110, 39)


,biosample_id,individual_id,biosample_name,notes,histological_diagnosis_id,histological_diagnosis_label,pathological_stage_id,pathological_stage_label,sample_origin_type_id,sample_origin_type_label,...,disease_id,disease_label,followup_days,followup_time_iso,followup_state_id,followup_state_label,onset_age_iso,onset_age_days,stage_id_individual,stage_label_individual
0,pgxbs-kftvhmjh,pgxind-kftx3l3h,37a8d3a5-baa0-427c-ab43-48ae4b93633b,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27975,Stage IA,OBI:0001479,specimen from organism,...,NCIT:C3512,Lung Adenocarcinoma,851.0,P28M,EFO:0030041,alive (follow-up status),P61Y6M7D,22469.0,NCIT:C27975,Stage IA
1,pgxbs-kftvhqda,pgxind-kftx3hag,5a109a3b-8edf-43f6-afb3-20e9ea68d617,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27975,Stage IA,OBI:0001479,specimen from organism,...,NCIT:C65197,Lung Adenocarcinoma with Mixed Bronchioloalveo...,1125.0,P37M,EFO:0030041,alive (follow-up status),P63Y1M13D,23053.0,NCIT:C27975,Stage IA
2,pgxbs-kftvi8g3,pgxind-kftx3is0,4bee0576-d35f-477b-b974-79cb89c88950,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27977,Stage IIIA,OBI:0001479,specimen from organism,...,NCIT:C3512,Lung Adenocarcinoma,30.0,P1M,EFO:0030041,alive (follow-up status),P75Y9M7D,27673.0,NCIT:C27977,Stage IIIA
3,pgxbs-kftvhmo2,pgxind-kftx3l8h,ef98c575-44c9-4524-8754-45250f60a5c8,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27971,Stage IV,OBI:0001479,specimen from organism,...,NCIT:C3512,Lung Adenocarcinoma,NaN,NaN,EFO:0030049,dead (follow-up status),P40Y2M10D,14680.0,NCIT:C27971,Stage IV
4,pgxbs-kftvhrxj,pgxind-kftx3pvf,81bfa60b-0e23-4d92-b82c-7a9c90e76fd7,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27975,Stage IA,OBI:0001479,specimen from organism,...,NCIT:C3512,Lung Adenocarcinoma,273.0,P9M,EFO:0030049,dead (follow-up status),P61Y4M11D,22412.0,NCIT:C27975,Stage IA


### Build Survival Outcome
Keep one biosample for each individual

In [11]:
survival_df = merged_meta_df.copy()

# Survival time
survival_df["time"] = pd.to_numeric(survival_df["followup_days"], errors="coerce")

def map_event(row):
    """
    Convert individual-level follow-up / vital status into a binary event label.
    0 = alive / censored
    1 = dead / event occurred
    """
    followup_state = str(row.get("followup_state_label", "")).strip().lower()
    vital_status = str(row.get("vital_status", "")).strip().upper()

    if "alive" in followup_state:
        return 0
    if "deceased" in followup_state or "dead" in followup_state:
        return 1

    if vital_status == "ALIVE":
        return 0
    if vital_status == "DECEASED":
        return 1

    return np.nan

survival_df["event"] = survival_df.apply(map_event, axis=1)

# Keep only rows with usable survival data
survival_df = survival_df.dropna(subset=["biosample_id", "individual_id", "time", "event"]).copy()
survival_df["event"] = survival_df["event"].astype(int)

print("Rows with usable survival data:", len(survival_df))
survival_df[[
    "biosample_id",
    "individual_id",
    "time",
    "event",
    "followup_state_label",
    "vital_status",
    "sex_label_individual"
]].head()

Rows with usable survival data: 859


,biosample_id,individual_id,time,event,followup_state_label,vital_status,sex_label_individual
0,pgxbs-kftvhmjh,pgxind-kftx3l3h,851.0,0,alive (follow-up status),ALIVE,female
1,pgxbs-kftvhqda,pgxind-kftx3hag,1125.0,0,alive (follow-up status),ALIVE,female
2,pgxbs-kftvi8g3,pgxind-kftx3is0,30.0,0,alive (follow-up status),ALIVE,female
4,pgxbs-kftvhrxj,pgxind-kftx3pvf,273.0,1,dead (follow-up status),DECEASED,female
6,pgxbs-kftvhvnr,pgxind-kftx3s6d,273.0,0,alive (follow-up status),ALIVE,male


In [12]:
survival_unique_df = (
    survival_df
    .groupby("individual_id")
    .sample(n=1, random_state=42)   
    .copy()
)

print("Rows after keeping one biosample per individual:", len(survival_unique_df))
survival_unique_df[[
    "biosample_id",
    "individual_id",
    "sample_origin_type_label",
    "time",
    "event",
    "sex_label_individual"
]].head()

Rows after keeping one biosample per individual: 391


,biosample_id,individual_id,sample_origin_type_label,time,event,sex_label_individual
402,pgxbs-kftvhrvw,pgxind-kftx3f83,specimen from organism,790.0,0,male
53,pgxbs-kftvhn58,pgxind-kftx3faj,specimen from organism,821.0,0,male
351,pgxbs-kftvhhlx,pgxind-kftx3fc3,specimen from organism,212.0,1,female
617,pgxbs-kftvi525,pgxind-kftx3fd3,specimen from organism,790.0,0,male
366,pgxbs-kftvhst1,pgxind-kftx3fd7,specimen from organism,456.0,0,female


In [13]:
FINAL_META_COLUMNS = [
    # biosample-level
    "biosample_id",
    "individual_id",
    "biosample_name",
    "notes",
    "histological_diagnosis_id",
    "histological_diagnosis_label",
    "pathological_stage_id",
    "pathological_stage_label",
    "sample_origin_type_id",
    "sample_origin_type_label",
    "sampled_tissue_id",
    "sampled_tissue_label",
    "tcgaproject_id",
    "tcgaproject_label",
    "icdo_morphology_id",
    "icdo_topography_id",
    "icdo_topography_label",

    # individual-level
    "sex_id_individual",
    "sex_label_individual",
    "vital_status",
    "info_age_at_diagnosis_days",
    "info_days_to_death",
    "info_death",
    "info_ethnicity",
    "info_race",
    "info_year_of_birth",
    "tcga_case_id",
    "tcga_submitter_id",
    "disease_id",
    "disease_label",
    "followup_days",
    "followup_time_iso",
    "followup_state_id",
    "followup_state_label",
    "onset_age_iso",
    "onset_age_days",
    "stage_id_individual",
    "stage_label_individual",

    # derived survival variables
    "time",
    "event",
]

FINAL_META_COLUMNS = [c for c in FINAL_META_COLUMNS if c in survival_unique_df.columns]

survival_unique_df = survival_unique_df[FINAL_META_COLUMNS].copy()

print("Final survival metadata shape:", survival_unique_df.shape)
survival_unique_df.head()

Final survival metadata shape: (391, 40)


,biosample_id,individual_id,biosample_name,notes,histological_diagnosis_id,histological_diagnosis_label,pathological_stage_id,pathological_stage_label,sample_origin_type_id,sample_origin_type_label,...,followup_days,followup_time_iso,followup_state_id,followup_state_label,onset_age_iso,onset_age_days,stage_id_individual,stage_label_individual,time,event
402,pgxbs-kftvhrvw,pgxind-kftx3f83,1a954c08-6297-4c7f-ad8c-e50cb2754f7e,Primary Tumor,NCIT:C2853,Papillary Adenocarcinoma,NCIT:C27967,Stage IIA,OBI:0001479,specimen from organism,...,790.0,P26M,EFO:0030041,alive (follow-up status),P50Y7M3D,18478.0,NCIT:C27967,Stage IIA,790.0,0
53,pgxbs-kftvhn58,pgxind-kftx3faj,301f0b81-a682-4d3e-a904-27b32381f43f,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27967,Stage IIA,OBI:0001479,specimen from organism,...,821.0,P27M,EFO:0030041,alive (follow-up status),P60Y0M24D,21938.0,NCIT:C27967,Stage IIA,821.0,0
351,pgxbs-kftvhhlx,pgxind-kftx3fc3,5d0ed8d1-6e84-4f6c-8e0e-0317aa8b0e3f,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27975,Stage IA,OBI:0001479,specimen from organism,...,212.0,P7M,EFO:0030049,dead (follow-up status),P61Y11M10D,22624.0,NCIT:C27975,Stage IA,212.0,1
617,pgxbs-kftvi525,pgxind-kftx3fd3,0c06cf85-61eb-4e6d-ae54-b8f5cbab9b63,Primary Tumor,NCIT:C65197,Lung Adenocarcinoma with Mixed Bronchioloalveo...,NCIT:C27976,Stage IB,OBI:0001479,specimen from organism,...,790.0,P26M,EFO:0030041,alive (follow-up status),P71Y4M29D,26082.0,NCIT:C27976,Stage IB,790.0,0
366,pgxbs-kftvhst1,pgxind-kftx3fd7,f0054242-3f54-41d4-9f74-94f61fefbfbc,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27976,Stage IB,OBI:0001479,specimen from organism,...,456.0,P15M,EFO:0030041,alive (follow-up status),P59Y3M13D,21653.0,NCIT:C27976,Stage IB,456.0,0


## Query gene-level CNV features with boolean Beacon requests

To keep the feature engineering simple, we use Boolean queries at the biosample level.

The idea is:

- `GENE_hit`: does this biosample contain at least one CNV event affecting the target gene?
- `GENE_gain_hit`: does this biosample contain at least one gain-type CNV event affecting the target gene?
- `GENE_loss_hit`: does this biosample contain at least one loss-type CNV event affecting the target gene?

With `requestedGranularity=boolean`, the Beacon endpoint returns a simple existence answer.


In [14]:
TARGET_GENE = "EGFR"
print("Target gene:", TARGET_GENE)

Target gene: EGFR


## Define Boolean query functions

We now define helper functions that query the Progenetix biosample-scoped CNV endpoint.

For each biosample, we query three Boolean features:

1. any CNV hit affecting the gene,
2. any gain event affecting the gene,
3. any loss event affecting the gene.

The returned value is converted into a binary feature:
- `1` if at least one matching event exists,
- `0` otherwise.

In [15]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})

def query_biosample_boolean(
    biosample_id,
    gene_symbol,
    extra_params=None,
    session=SESSION,
    timeout=120
):
    url = f"https://progenetix.org/beacon/biosamples/{biosample_id}/g_variants/"

    params = {
        "geneId": gene_symbol,
        "requestedGranularity": "boolean",
    }

    if extra_params:
        params.update(extra_params)

    response = session.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    data = response.json()

    exists = data.get("responseSummary", {}).get("exists", False)
    return int(bool(exists))

## Define simplified Boolean CNV states

To keep the workflow simple, we collapse CNV states into two broader categories:

- **gain** = low-level gain or high-level gain
- **loss** = low-level loss or high-level loss

This allows us to define three compact Boolean features for the target gene:

- `GENE_hit`
- `GENE_gain_hit`
- `GENE_loss_hit`

This simplified design is easier to interpret than keeping separate high- and low-level states.

In [16]:
# CNV state groups
GAIN_STATE_IDS = ["EFO:0030071", "EFO:0030072"]   # low-level gain, high-level gain
LOSS_STATE_IDS = ["EFO:0030068", "EFO:0020073"]   # low-level loss, high-level loss

print("GAIN_STATE_IDS =", GAIN_STATE_IDS)
print("LOSS_STATE_IDS =", LOSS_STATE_IDS)

GAIN_STATE_IDS = ['EFO:0030071', 'EFO:0030072']
LOSS_STATE_IDS = ['EFO:0030068', 'EFO:0020073']


In [17]:
def make_gene_hit_params():
    # Boolean query for any CNV event affecting the target gene.
    return {}


def make_gene_gain_param_list():
    # Boolean query candidates for gain-related events.
    # One query per gain state ID.
    return [{"filters": state_id} for state_id in GAIN_STATE_IDS]


def make_gene_loss_param_list():
    # Boolean query candidates for loss-related events.
    # One query per loss state ID.
    return [{"filters": state_id} for state_id in LOSS_STATE_IDS]

def query_biosample_boolean(
    biosample_id,
    gene_symbol,
    extra_params=None,
    session=SESSION,
    timeout=120
):
    # Run a Boolean Beacon query against one biosample-scoped g_variants endpoint.
    url = f"https://progenetix.org/beacon/biosamples/{biosample_id}/g_variants/"

    params = {
        "geneId": gene_symbol,
        "requestedGranularity": "boolean",
    }

    if extra_params:
        params.update(extra_params)

    response = session.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    data = response.json()

    exists = data.get("responseSummary", {}).get("exists", False)
    return int(bool(exists))

In [18]:
def query_any_of_param_list(biosample_id, gene_symbol, param_list):
    # Return 1 if any Boolean query in param_list returns True, else 0.
    for params in param_list:
        try:
            hit = query_biosample_boolean(
                biosample_id=biosample_id,
                gene_symbol=gene_symbol,
                extra_params=params
            )
            if hit == 1:
                return 1
        except Exception:
            continue
    return 0

In [19]:
def build_boolean_gene_features(
    biosample_ids,
    gene_symbol
):
    """
    Build three Boolean CNV features for one gene:
    - gene hit
    - gene gain hit
    - gene loss hit
    """
    rows = []
    failures = []

    gain_param_list = make_gene_gain_param_list()
    loss_param_list = make_gene_loss_param_list()

    for biosample_id in tqdm(biosample_ids, desc=f"Querying Boolean CNV features for {gene_symbol}"):
        try:
            gene_hit = query_biosample_boolean(
                biosample_id=biosample_id,
                gene_symbol=gene_symbol,
                extra_params=make_gene_hit_params()
            )

            gene_gain_hit = query_any_of_param_list(
                biosample_id=biosample_id,
                gene_symbol=gene_symbol,
                param_list=gain_param_list
            )

            gene_loss_hit = query_any_of_param_list(
                biosample_id=biosample_id,
                gene_symbol=gene_symbol,
                param_list=loss_param_list
            )

            rows.append({
                "biosample_id": biosample_id,
                f"{gene_symbol}_hit": gene_hit,
                f"{gene_symbol}_gain_hit": gene_gain_hit,
                f"{gene_symbol}_loss_hit": gene_loss_hit,
            })

        except Exception as e:
            failures.append((biosample_id, str(e)))
            rows.append({
                "biosample_id": biosample_id,
                f"{gene_symbol}_hit": np.nan,
                f"{gene_symbol}_gain_hit": np.nan,
                f"{gene_symbol}_loss_hit": np.nan,
            })

    feature_df = pd.DataFrame(rows)
    return feature_df, failures

In [20]:
TARGET_GENE = "EGFR"

biosample_ids_for_features = survival_unique_df["biosample_id"].dropna().astype(str).tolist()

gene_feature_df, gene_feature_failures = build_boolean_gene_features(
    biosample_ids=biosample_ids_for_features,
    gene_symbol=TARGET_GENE,
)

print("Feature table shape:", gene_feature_df.shape)
print("Number of failed biosample queries:", len(gene_feature_failures))
gene_feature_df.head()

Querying Boolean CNV features for EGFR:   0%|          | 0/391 [00:00<?, ?it/s]

Feature table shape: (391, 4)
Number of failed biosample queries: 0


,biosample_id,EGFR_hit,EGFR_gain_hit,EGFR_loss_hit
0,pgxbs-kftvhrvw,1,1,1
1,pgxbs-kftvhn58,1,1,1
2,pgxbs-kftvhhlx,1,1,1
3,pgxbs-kftvi525,1,1,1
4,pgxbs-kftvhst1,1,1,1


In [21]:
analysis_df = survival_unique_df.merge(
    gene_feature_df,
    on="biosample_id",
    how="left"
)

feature_cols = [
    f"{TARGET_GENE}_hit",
    f"{TARGET_GENE}_gain_hit",
    f"{TARGET_GENE}_loss_hit",
]

analysis_df[feature_cols] = analysis_df[feature_cols].fillna(0).astype(int)

analysis_df.head()

,biosample_id,individual_id,biosample_name,notes,histological_diagnosis_id,histological_diagnosis_label,pathological_stage_id,pathological_stage_label,sample_origin_type_id,sample_origin_type_label,...,followup_state_label,onset_age_iso,onset_age_days,stage_id_individual,stage_label_individual,time,event,EGFR_hit,EGFR_gain_hit,EGFR_loss_hit
0,pgxbs-kftvhrvw,pgxind-kftx3f83,1a954c08-6297-4c7f-ad8c-e50cb2754f7e,Primary Tumor,NCIT:C2853,Papillary Adenocarcinoma,NCIT:C27967,Stage IIA,OBI:0001479,specimen from organism,...,alive (follow-up status),P50Y7M3D,18478.0,NCIT:C27967,Stage IIA,790.0,0,1,1,1
1,pgxbs-kftvhn58,pgxind-kftx3faj,301f0b81-a682-4d3e-a904-27b32381f43f,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27967,Stage IIA,OBI:0001479,specimen from organism,...,alive (follow-up status),P60Y0M24D,21938.0,NCIT:C27967,Stage IIA,821.0,0,1,1,1
2,pgxbs-kftvhhlx,pgxind-kftx3fc3,5d0ed8d1-6e84-4f6c-8e0e-0317aa8b0e3f,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27975,Stage IA,OBI:0001479,specimen from organism,...,dead (follow-up status),P61Y11M10D,22624.0,NCIT:C27975,Stage IA,212.0,1,1,1,1
3,pgxbs-kftvi525,pgxind-kftx3fd3,0c06cf85-61eb-4e6d-ae54-b8f5cbab9b63,Primary Tumor,NCIT:C65197,Lung Adenocarcinoma with Mixed Bronchioloalveo...,NCIT:C27976,Stage IB,OBI:0001479,specimen from organism,...,alive (follow-up status),P71Y4M29D,26082.0,NCIT:C27976,Stage IB,790.0,0,1,1,1
4,pgxbs-kftvhst1,pgxind-kftx3fd7,f0054242-3f54-41d4-9f74-94f61fefbfbc,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27976,Stage IB,OBI:0001479,specimen from organism,...,alive (follow-up status),P59Y3M13D,21653.0,NCIT:C27976,Stage IB,456.0,0,1,1,1


In [22]:
for col in feature_cols:
    print(f"\n{col}")
    print(analysis_df[col].value_counts(dropna=False))


EGFR_hit
EGFR_hit
1    391
Name: count, dtype: int64

EGFR_gain_hit
EGFR_gain_hit
1    391
Name: count, dtype: int64

EGFR_loss_hit
EGFR_loss_hit
1    391
Name: count, dtype: int64
